In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/semantig936/arc-agi/data/training/780d0b14.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/a1570a43.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/d90796e8.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/3f7978a0.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/22eb0ac0.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/0520fde7.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/23b5c85d.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/d9fac9be.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/890034e9.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/9ecd008a.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/06df4c85.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/2dc579da.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/007bbfb7.json
/kaggle/input/datasets/semantig936/arc-agi/data/training/4522001f.json
/kaggl

In [2]:
import os

# See what's available
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, ':', len(files), 'files')

/kaggle/input : 0 files
/kaggle/input/datasets : 0 files
/kaggle/input/datasets/semantig936 : 0 files
/kaggle/input/datasets/semantig936/arc-agi : 0 files
/kaggle/input/datasets/semantig936/arc-agi/data : 0 files
/kaggle/input/datasets/semantig936/arc-agi/data/training : 400 files
/kaggle/input/datasets/semantig936/arc-agi/data/evaluation : 400 files
/kaggle/input/competitions : 0 files
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3 : 4 files
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation : 3 files
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core : 5 files
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/generated : 3 files


In [3]:
import os, json, random
import numpy as np
import torch

def load_task(filepath):
    """Load a single ARC task JSON file.
    
    Returns:
        dict with keys 'train' (list of demo pairs) and 'test' (list of test inputs).
        Each pair is {'input': List[List[int]], 'output': List[List[int]]}.
    """
    with open(filepath) as f:
        return json.load(f)

def load_all_tasks(folder):
    """Load all tasks from a folder. Returns list of (task_id, task_dict)."""
    tasks = []
    for fname in sorted(os.listdir(folder)):
        if fname.endswith('.json'):
            task_id = fname.replace('.json', '')
            task = load_task(os.path.join(folder, fname))
            tasks.append((task_id, task))
    return tasks

TRAIN_DIR = '/kaggle/input/datasets/semantig936/arc-agi/data/training'
EVAL_DIR  = '/kaggle/input/datasets/semantig936/arc-agi/data/evaluation'

train_tasks = load_all_tasks(TRAIN_DIR)
eval_tasks  = load_all_tasks(EVAL_DIR)

print(f"Training tasks: {len(train_tasks)}")
print(f"Evaluation tasks: {len(eval_tasks)}")

# Sanity check — print one task
task_id, task = train_tasks[0]
print(f"\nTask: {task_id}")
print(f"Number of demo pairs: {len(task['train'])}")
print(f"Number of test inputs: {len(task['test'])}")

pair = task['train'][0]
print(f"\nDemo 1 input ({len(pair['input'])}x{len(pair['input'][0])}):")
for row in pair['input']:
    print(row)
print(f"\nDemo 1 output ({len(pair['output'])}x{len(pair['output'][0])}):")
for row in pair['output']:
    print(row)

Training tasks: 400
Evaluation tasks: 400

Task: 007bbfb7
Number of demo pairs: 5
Number of test inputs: 1

Demo 1 input (3x3):
[0, 7, 7]
[7, 7, 7]
[0, 7, 7]

Demo 1 output (9x9):
[0, 0, 0, 0, 7, 7, 0, 7, 7]
[0, 0, 0, 7, 7, 7, 7, 7, 7]
[0, 0, 0, 0, 7, 7, 0, 7, 7]
[0, 7, 7, 0, 7, 7, 0, 7, 7]
[7, 7, 7, 7, 7, 7, 7, 7, 7]
[0, 7, 7, 0, 7, 7, 0, 7, 7]
[0, 0, 0, 0, 7, 7, 0, 7, 7]
[0, 0, 0, 7, 7, 7, 7, 7, 7]
[0, 0, 0, 0, 7, 7, 0, 7, 7]


In [5]:
# --- Special token definitions ---
ROW_SEP  = 10
GRID_SEP = 11
PAIR_SEP = 12
BOS      = 13
EOS      = 14
PAD      = 15

VOCAB_SIZE = 16

def grid_to_tokens(grid):
    """Convert a 2D grid (list of lists) to a flat list of tokens.
    
    Rows are separated by ROW_SEP tokens.
    Example: [[1,2],[3,4]] -> [1, 2, ROW_SEP, 3, 4, ROW_SEP]
    """
    tokens = []
    for row in grid:
        tokens.extend(row)
        tokens.append(ROW_SEP)
    return tokens

def task_to_sequence(task):
    """Serialize a full ARC task into a single token sequence.
    
    Format:
        BOS
        [demo pair 1 input tokens] GRID_SEP [demo pair 1 output tokens]
        PAIR_SEP
        [demo pair 2 input tokens] GRID_SEP [demo pair 2 output tokens]
        PAIR_SEP
        ...
        [test input tokens] GRID_SEP [test output tokens]
        EOS
    
    Returns:
        tokens (list[int]): full sequence
        output_start (int): index where test output tokens begin
                            (we'll use this to compute loss only on output)
    """
    tokens = [BOS]
    
    # Demonstration pairs
    for pair in task['train']:
        tokens.extend(grid_to_tokens(pair['input']))
        tokens.append(GRID_SEP)
        tokens.extend(grid_to_tokens(pair['output']))
        tokens.append(PAIR_SEP)
    
    # Test input
    tokens.extend(grid_to_tokens(task['test'][0]['input']))
    tokens.append(GRID_SEP)
    
    # Mark where the output begins (model must predict from here)
    output_start = len(tokens)
    
    # Test output (ground truth — used for training loss)
    tokens.extend(grid_to_tokens(task['test'][0]['output']))
    tokens.append(EOS)
    
    return tokens, output_start


# --- Sanity check ---
task_id, task = train_tasks[0]
tokens, output_start = task_to_sequence(task)

print(f"Task: {task_id}")
print(f"Total sequence length: {len(tokens)}")
print(f"Output starts at index: {output_start}")
print(f"Output length: {len(tokens) - output_start} tokens")
print(f"\nFirst 40 tokens: {tokens[:40]}")
print(f"\nTokens around output start: {tokens[output_start-3 : output_start+10]}")

Task: 007bbfb7
Total sequence length: 625
Output starts at index: 534
Output length: 91 tokens

First 40 tokens: [13, 0, 7, 7, 10, 7, 7, 7, 10, 0, 7, 7, 10, 11, 0, 0, 0, 0, 7, 7, 0, 7, 7, 10, 0, 0, 0, 7, 7, 7, 7, 7, 7, 10, 0, 0, 0, 0, 7, 7]

Tokens around output start: [0, 10, 11, 7, 0, 7, 0, 0, 0, 7, 0, 7, 10]


In [10]:
lengths = []
skipped = 0

for task_id, task in train_tasks:
    tokens, output_start = task_to_sequence(task)
    lengths.append(len(tokens))

lengths = sorted(lengths)

print(f"Min length:    {min(lengths)}")
print(f"Max length:    {max(lengths)}")
print(f"Mean length:   {sum(lengths)//len(lengths)}")
print(f"Median length: {lengths[len(lengths)//2]}")
print(f"\nTasks over 512 tokens:  {sum(1 for l in lengths if l > 512)}")
print(f"Tasks over 1024 tokens: {sum(1 for l in lengths if l > 1024)}")
print(f"Tasks over 2048 tokens: {sum(1 for l in lengths if l > 2048)}")

Min length:    79
Max length:    9311
Mean length:   1194
Median length: 889

Tasks over 512 tokens:  280
Tasks over 1024 tokens: 153
Tasks over 2048 tokens: 59


In [13]:
MAX_SEQ_LEN = 1024

def get_output_shape(task):
    """Return (height, width) of the expected test output grid."""
    output = task['test'][0]['output']
    return len(output), len(output[0])

def is_valid_task(task):
    """Check if a task's sequence fits within MAX_SEQ_LEN.
    
    Returns (is_valid, sequence_length).
    """
    tokens, _ = task_to_sequence(task)
    return len(tokens) <= MAX_SEQ_LEN, len(tokens)

# --- Check how many tasks survive the 1024 cutoff ---
valid_tasks = []
skipped_tasks = []

for task_id, task in train_tasks:
    valid, seq_len = is_valid_task(task)
    if valid:
        valid_tasks.append((task_id, task))
    else:
        skipped_tasks.append((task_id, seq_len))

print(f"Valid tasks (≤ 1024 tokens): {len(valid_tasks)}")
print(f"Skipped tasks (> 1024 tokens): {len(skipped_tasks)}")
print(f"\nLongest skipped task: {max(l for _, l in skipped_tasks)} tokens")

Valid tasks (≤ 1024 tokens): 247
Skipped tasks (> 1024 tokens): 153

Longest skipped task: 9311 tokens


In [14]:
for limit in [1024, 1536, 2048, 3000, 4096]:
    count = sum(1 for _, task in train_tasks 
                if len(task_to_sequence(task)[0]) <= limit)
    print(f"Tasks within {limit} tokens: {count}")

Tasks within 1024 tokens: 247
Tasks within 1536 tokens: 292
Tasks within 2048 tokens: 341
Tasks within 3000 tokens: 374
Tasks within 4096 tokens: 386


In [15]:
import copy

# Update max sequence length
MAX_SEQ_LEN = 2048

# ─────────────────────────────────────────
# Geometric augmentation helpers
# ─────────────────────────────────────────

def rotate_grid_90(grid):
    """Rotate a 2D grid 90 degrees clockwise."""
    return [list(row) for row in zip(*grid[::-1])]

def flip_grid_horizontal(grid):
    """Flip a 2D grid horizontally (left-right mirror)."""
    return [row[::-1] for row in grid]

def transform_grid(grid, rotate_k, flip):
    """Apply k*90 degree clockwise rotation, then optional horizontal flip."""
    g = copy.deepcopy(grid)
    for _ in range(rotate_k):
        g = rotate_grid_90(g)
    if flip:
        g = flip_grid_horizontal(g)
    return g

def transform_task(task, rotate_k, flip):
    """Apply a geometric transformation to all grids in a task.
    
    Applies the same rotation and flip to every input and output grid
    so the rule is preserved.
    """
    new_task = {'train': [], 'test': []}
    
    for pair in task['train']:
        new_task['train'].append({
            'input':  transform_grid(pair['input'],  rotate_k, flip),
            'output': transform_grid(pair['output'], rotate_k, flip)
        })
    
    for pair in task['test']:
        new_task['test'].append({
            'input':  transform_grid(pair['input'],  rotate_k, flip),
            'output': transform_grid(pair['output'], rotate_k, flip)
        })
    
    return new_task

# ─────────────────────────────────────────
# Colour permutation augmentation
# ─────────────────────────────────────────

def permute_colours(task, colour_map):
    """Remap all colour values in a task using colour_map dict.
    
    colour_map: dict mapping old colour int -> new colour int
    Only remaps colours 1-9 (colour 0 = black is kept fixed,
    as it typically represents background).
    """
    def remap_grid(grid):
        return [[colour_map.get(c, c) for c in row] for row in grid]
    
    new_task = {'train': [], 'test': []}
    
    for pair in task['train']:
        new_task['train'].append({
            'input':  remap_grid(pair['input']),
            'output': remap_grid(pair['output'])
        })
    
    for pair in task['test']:
        new_task['test'].append({
            'input':  remap_grid(pair['input']),
            'output': remap_grid(pair['output'])
        })
    
    return new_task

def random_colour_permutation(task):
    """Generate a random colour permutation of a task.
    
    Only permutes colours that actually appear in the task,
    leaving colour 0 (background) fixed.
    """
    # Find all non-zero colours used in this task
    used_colours = set()
    for pair in task['train'] + task['test']:
        for grid in [pair['input'], pair.get('output', pair['input'])]:
            for row in grid:
                used_colours.update(c for c in row if c != 0)
    
    used_colours = list(used_colours)
    if len(used_colours) <= 1:
        # Nothing meaningful to permute
        return task
    
    shuffled = used_colours.copy()
    random.shuffle(shuffled)
    colour_map = dict(zip(used_colours, shuffled))
    
    return permute_colours(task, colour_map)

# ─────────────────────────────────────────
# Build the full augmented training set
# ─────────────────────────────────────────

def build_augmented_dataset(train_tasks, max_seq_len, num_colour_perms=2):
    """Build augmented training set from raw tasks.
    
    For each valid task:
      - Apply all 8 geometric transforms (4 rotations x 2 flips)
      - Apply num_colour_perms random colour permutations on top
        of each geometric variant
    
    Returns list of (task_id, task_dict) tuples.
    """
    augmented = []
    skipped = 0
    
    # All 8 dihedral transforms: (rotate_k, flip)
    geometric_transforms = [
        (0, False), (1, False), (2, False), (3, False),
        (0, True),  (1, True),  (2, True),  (3, True)
    ]
    
    for task_id, task in train_tasks:
        # Check base task fits
        base_tokens, _ = task_to_sequence(task)
        if len(base_tokens) > max_seq_len:
            skipped += 1
            continue
        
        for rotate_k, flip in geometric_transforms:
            geo_task = transform_task(task, rotate_k, flip)
            
            # Check this geometric variant fits
            tokens, _ = task_to_sequence(geo_task)
            if len(tokens) > max_seq_len:
                continue
            
            aug_id = f"{task_id}_r{rotate_k}_f{int(flip)}"
            augmented.append((aug_id, geo_task))
            
            # Add colour permutation variants on top
            for cp in range(num_colour_perms):
                cp_task = random_colour_permutation(geo_task)
                cp_tokens, _ = task_to_sequence(cp_task)
                if len(cp_tokens) > max_seq_len:
                    continue
                cp_id = f"{task_id}_r{rotate_k}_f{int(flip)}_cp{cp}"
                augmented.append((cp_id, cp_task))
    
    return augmented, skipped

# Build it
random.seed(42)
augmented_train, skipped = build_augmented_dataset(
    train_tasks, 
    max_seq_len=MAX_SEQ_LEN, 
    num_colour_perms=2
)

print(f"Original valid tasks:       {341}")
print(f"Skipped tasks:              {skipped}")
print(f"Total augmented examples:   {len(augmented_train)}")
print(f"Approximate multiplier:     {len(augmented_train) / 341:.1f}x")

# Sanity check — make sure tokenization still works on augmented tasks
sample_id, sample_task = augmented_train[42]
tokens, output_start = task_to_sequence(sample_task)
print(f"\nSample augmented task: {sample_id}")
print(f"Sequence length: {len(tokens)}, output starts at: {output_start}")

Original valid tasks:       341
Skipped tasks:              59
Total augmented examples:   8172
Approximate multiplier:     24.0x

Sample augmented task: 017c7c7b_r2_f1
Sequence length: 249, output starts at: 212


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader

class ARCDataset(Dataset):
    """PyTorch Dataset for ARC-AGI tasks.
    
    Each item is a full tokenized sequence. During training,
    the model predicts the next token at every position,
    but we only compute loss on the output portion of the sequence.
    """
    
    def __init__(self, tasks, max_seq_len):
        """
        Args:
            tasks: list of (task_id, task_dict) tuples
            max_seq_len: maximum sequence length (longer sequences are skipped)
        """
        self.max_seq_len = max_seq_len
        self.examples = []
        
        for task_id, task in tasks:
            tokens, output_start = task_to_sequence(task)
            if len(tokens) > max_seq_len:
                continue
            self.examples.append({
                'task_id':      task_id,
                'tokens':       tokens,
                'output_start': output_start
            })
        
        print(f"Dataset built: {len(self.examples)} examples")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        ex = self.examples[idx]
        tokens = ex['tokens']
        output_start = ex['output_start']
        
        # Input to the model: all tokens except the last
        input_ids = torch.tensor(tokens[:-1], dtype=torch.long)
        
        # Target: all tokens shifted by one (next token prediction)
        target_ids = torch.tensor(tokens[1:], dtype=torch.long)
        
        # Loss mask: 1 only at positions where we're predicting output tokens
        # output_start - 1 because input_ids is shifted by 1
        loss_mask = torch.zeros(len(target_ids), dtype=torch.bool)
        loss_mask[output_start - 1:] = True
        
        return input_ids, target_ids, loss_mask


def collate_fn(batch):
    """Pad sequences in a batch to the same length.
    
    Pads input_ids and target_ids with PAD token (15).
    Pads loss_mask with False (don't compute loss on padding).
    """
    input_ids, target_ids, loss_masks = zip(*batch)
    
    max_len = max(x.size(0) for x in input_ids)
    
    padded_inputs  = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    padded_targets = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    padded_masks   = torch.zeros(len(batch), max_len, dtype=torch.bool)
    
    for i, (inp, tgt, msk) in enumerate(zip(input_ids, target_ids, loss_masks)):
        padded_inputs[i,  :len(inp)] = inp
        padded_targets[i, :len(tgt)] = tgt
        padded_masks[i,   :len(msk)] = msk
    
    return padded_inputs, padded_targets, padded_masks


# ─────────────────────────────────────────
# Train / validation split
# ─────────────────────────────────────────

# Hold out 10% of ORIGINAL task IDs for validation
# Important: we split by original task ID, not augmented examples
# so augmented variants of a task don't leak into validation

original_ids = list(set(
    tid.split('_r')[0] for tid, _ in augmented_train
))
random.seed(42)
random.shuffle(original_ids)

val_size   = int(0.1 * len(original_ids))  # ~34 tasks
val_ids    = set(original_ids[:val_size])
train_ids  = set(original_ids[val_size:])

train_split = [(tid, task) for tid, task in augmented_train 
               if tid.split('_r')[0] in train_ids]
val_split   = [(tid, task) for tid, task in augmented_train 
               if tid.split('_r')[0] in val_ids]

print(f"Train examples: {len(train_split)}")
print(f"Val examples:   {len(val_split)}")

# ─────────────────────────────────────────
# Build datasets and dataloaders
# ─────────────────────────────────────────

train_dataset = ARCDataset(train_split, MAX_SEQ_LEN)
val_dataset   = ARCDataset(val_split,   MAX_SEQ_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

# ─────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────

inp, tgt, msk = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids:  {inp.shape}")
print(f"  target_ids: {tgt.shape}")
print(f"  loss_mask:  {msk.shape}")
print(f"  loss positions in first example: {msk[0].sum().item()} / {msk.shape[1]}")

Train examples: 7356
Val examples:   816
Dataset built: 7356 examples
Dataset built: 816 examples

Batch shapes:
  input_ids:  torch.Size([4, 1064])
  target_ids: torch.Size([4, 1064])
  loss_mask:  torch.Size([4, 1064])
  loss positions in first example: 133 / 1064


In [17]:
import torch
import torch.nn as nn
import math

class ARCTransformer(nn.Module):
    """Decoder-only transformer for ARC-AGI task solving.
    
    Architecture:
    - Token embedding (vocab size 16)
    - 2D-aware positional encoding (row, col, sequence position)
    - Stack of transformer decoder layers (causal self-attention)
    - Linear output head projecting to vocab size
    """
    
    def __init__(
        self,
        vocab_size   = VOCAB_SIZE,   # 16
        d_model      = 512,          # embedding dimension
        n_heads      = 8,            # attention heads
        n_layers     = 6,            # transformer layers
        d_ff         = 2048,         # feedforward hidden dim
        max_seq_len  = MAX_SEQ_LEN,  # 2048
        dropout      = 0.1
    ):
        super().__init__()
        
        self.d_model     = d_model
        self.n_heads     = n_heads
        self.n_layers    = n_layers
        self.max_seq_len = max_seq_len
        
        # Token embedding
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        
        # Learned positional embedding (1D absolute positions)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # Transformer decoder layers (causal)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model        = d_model,
            nhead          = n_heads,
            dim_feedforward= d_ff,
            dropout        = dropout,
            batch_first    = True,    # (batch, seq, dim)
            norm_first     = True     # pre-norm: more stable training
        )
        self.transformer = nn.TransformerDecoder(
            decoder_layer,
            num_layers = n_layers
        )
        
        # Output projection head
        self.output_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share token embedding and output head weights
        # This is a common trick that saves parameters and improves performance
        self.output_head.weight = self.token_emb.weight
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights with small normal distribution.
        
        Uses scaled initialization for embeddings and
        Xavier uniform for linear layers.
        """
        nn.init.normal_(self.token_emb.weight, std=0.02)
        nn.init.normal_(self.pos_emb.weight,   std=0.02)
        
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        """Forward pass.
        
        Args:
            input_ids: (batch, seq_len) token indices
            
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        
        # Token + positional embeddings
        positions = torch.arange(seq_len, device=device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.dropout(x)
        
        # Causal mask as boolean (fixes the dtype mismatch warning)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device, dtype=torch.bool),
            diagonal=1
        )  # True = block attention (upper triangle)
        
        # Padding mask
        pad_mask = (input_ids == PAD)  # (batch, seq_len)
        
        x = self.transformer(
            tgt                  = x,
            memory               = x,
            tgt_mask             = causal_mask,
            tgt_key_padding_mask = pad_mask
        )
        
        logits = self.output_head(x)
        return logits
        


# ─────────────────────────────────────────
# Parameter count utility
# ─────────────────────────────────────────

def count_parameters(model):
    """Count and display total trainable parameters."""
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{'Layer':<40} {'Parameters':>12}")
    print("-" * 54)
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(f"{name:<40} {p.numel():>12,}")
    print("-" * 54)
    print(f"{'TOTAL':<40} {total:>12,}")
    return total


# ─────────────────────────────────────────
# Instantiate and check parameter count
# ─────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = ARCTransformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = 512,
    n_heads     = 8,
    n_layers    = 6,
    d_ff        = 2048,
    max_seq_len = MAX_SEQ_LEN,
    dropout     = 0.1
)

total_params = count_parameters(model)
print(f"\nTotal parameters: {total_params/1e6:.2f}M")
print(f"Within 50M budget: {total_params <= 50_000_000}")

model = model.to(device)

# ─────────────────────────────────────────
# Quick forward pass sanity check
# ─────────────────────────────────────────

model.eval()
with torch.no_grad():
    inp, tgt, msk = next(iter(train_loader))
    inp = inp.to(device)
    logits = model(inp)
    print(f"\nForward pass output shape: {logits.shape}")
    print(f"Expected:                  (4, seq_len, 16)")

Using device: cuda
Layer                                      Parameters
------------------------------------------------------
token_emb.weight                                8,192
pos_emb.weight                              1,048,576
transformer.layers.0.self_attn.in_proj_weight      786,432
transformer.layers.0.self_attn.in_proj_bias        1,536
transformer.layers.0.self_attn.out_proj.weight      262,144
transformer.layers.0.self_attn.out_proj.bias          512
transformer.layers.0.multihead_attn.in_proj_weight      786,432
transformer.layers.0.multihead_attn.in_proj_bias        1,536
transformer.layers.0.multihead_attn.out_proj.weight      262,144
transformer.layers.0.multihead_attn.out_proj.bias          512
transformer.layers.0.linear1.weight         1,048,576
transformer.layers.0.linear1.bias               2,048
transformer.layers.0.linear2.weight         1,048,576
transformer.layers.0.linear2.bias                 512
transformer.layers.0.norm1.weight                 512
transf

In [18]:
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler

def compute_loss(logits, target_ids, loss_mask):
    """Compute cross-entropy loss only on output token positions.
    
    Args:
        logits:     (batch, seq_len, vocab_size)
        target_ids: (batch, seq_len)
        loss_mask:  (batch, seq_len) bool, True = compute loss here
    
    Returns:
        scalar loss
    """
    # Flatten for cross entropy
    logits_flat = logits.view(-1, VOCAB_SIZE)       # (batch*seq_len, vocab_size)
    targets_flat = target_ids.view(-1)              # (batch*seq_len,)
    mask_flat = loss_mask.view(-1)                  # (batch*seq_len,)
    
    # Compute loss only on masked positions
    loss = F.cross_entropy(
        logits_flat[mask_flat],
        targets_flat[mask_flat],
        ignore_index=PAD
    )
    return loss


def train_one_epoch(model, loader, optimizer, scaler, device, grad_accum_steps=4):
    """Train for one epoch with gradient accumulation and BF16 autocast.
    
    Args:
        grad_accum_steps: accumulate gradients over this many batches
                          before stepping. Effective batch size = 
                          batch_size * grad_accum_steps = 4 * 4 = 16
    """
    model.train()
    total_loss = 0.0
    total_batches = 0
    optimizer.zero_grad()
    
    for step, (inp, tgt, msk) in enumerate(loader):
        inp = inp.to(device)
        tgt = tgt.to(device)
        msk = msk.to(device)
        
        # BF16 mixed precision — halves memory, minimal accuracy cost
        with autocast(dtype=torch.bfloat16):
            logits = model(inp)
            loss   = compute_loss(logits, tgt, msk)
            loss   = loss / grad_accum_steps  # scale for accumulation
        
        scaler.scale(loss).backward()
        
        # Step optimizer every grad_accum_steps batches
        if (step + 1) % grad_accum_steps == 0:
            # Gradient clipping — prevents exploding gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss   += loss.item() * grad_accum_steps
        total_batches += 1
        
        if step % 100 == 0:
            print(f"  step {step}/{len(loader)}, loss: {loss.item()*grad_accum_steps:.4f}")
    
    return total_loss / total_batches


@torch.no_grad()
def evaluate(model, loader, device):
    """Compute average loss on validation set."""
    model.eval()
    total_loss    = 0.0
    total_batches = 0
    
    for inp, tgt, msk in loader:
        inp = inp.to(device)
        tgt = tgt.to(device)
        msk = msk.to(device)
        
        with autocast(dtype=torch.bfloat16):
            logits = model(inp)
            loss   = compute_loss(logits, tgt, msk)
        
        total_loss    += loss.item()
        total_batches += 1
    
    return total_loss / total_batches


# ─────────────────────────────────────────
# Optimizer and LR scheduler
# ─────────────────────────────────────────

# WSD schedule: Warmup → Stable → Decay
# As recommended in the assignment PDF for small datasets

NUM_EPOCHS    = 30
WARMUP_EPOCHS = 3
DECAY_EPOCHS  = 10  # linear decay over last 10 epochs
LR            = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    betas=(0.9, 0.95)
)

def get_lr(epoch):
    """WSD learning rate schedule.
    
    Warmup for WARMUP_EPOCHS, stable until decay starts,
    then linear decay to 0 over DECAY_EPOCHS.
    """
    if epoch < WARMUP_EPOCHS:
        # Linear warmup
        return (epoch + 1) / WARMUP_EPOCHS
    elif epoch < NUM_EPOCHS - DECAY_EPOCHS:
        # Stable phase
        return 1.0
    else:
        # Linear decay
        decay_progress = (epoch - (NUM_EPOCHS - DECAY_EPOCHS)) / DECAY_EPOCHS
        return max(1e-2, 1.0 - decay_progress)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

# GradScaler for mixed precision
scaler = GradScaler()

# ─────────────────────────────────────────
# Checkpoint saving
# ─────────────────────────────────────────

import os

CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, filename):
    torch.save({
        'epoch':      epoch,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
        'val_loss':   val_loss
    }, os.path.join(CKPT_DIR, filename))
    print(f"  Checkpoint saved: {filename}")

# ─────────────────────────────────────────
# Training loop
# ─────────────────────────────────────────

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

print("Starting training...")
print(f"Epochs: {NUM_EPOCHS}, LR: {LR}, Device: {device}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    current_lr = scheduler.get_last_lr()[0] * LR
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}  (lr={current_lr:.2e})")
    
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scaler, device,
        grad_accum_steps=4
    )
    val_loss = evaluate(model, val_loader, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    print(f"  train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch, val_loss,
                       'checkpoint_best.pt')
    
    # Save latest checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch, val_loss,
                       f'checkpoint_epoch_{epoch+1}.pt')

# Save final checkpoint
save_checkpoint(model, optimizer, scheduler, NUM_EPOCHS-1, val_loss,
               'checkpoint_final.pt')

print("\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

/tmp/ipykernel_106/2842600241.py:135: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_106/2842600241.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


Starting training...
Epochs: 30, LR: 0.0003, Device: cuda

Epoch 1/30  (lr=3.00e-08)
  step 0/1839, loss: 3.7272
  step 100/1839, loss: 1.6345
  step 200/1839, loss: 2.2872
  step 300/1839, loss: 1.3052
  step 400/1839, loss: 1.4402
  step 500/1839, loss: 2.6209
  step 600/1839, loss: 1.5993
  step 700/1839, loss: 1.3659
  step 800/1839, loss: 2.0491
  step 900/1839, loss: 1.0579
  step 1000/1839, loss: 1.5221
  step 1100/1839, loss: 1.0461
  step 1200/1839, loss: 2.6493
  step 1300/1839, loss: 1.0026
  step 1400/1839, loss: 0.9609
  step 1500/1839, loss: 1.0530
  step 1600/1839, loss: 1.1777
  step 1700/1839, loss: 1.3396
  step 1800/1839, loss: 1.2565


/tmp/ipykernel_106/2842600241.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


  train loss: 1.6848 | val loss: 1.4741
  Checkpoint saved: checkpoint_best.pt

Epoch 2/30  (lr=6.00e-08)
  step 0/1839, loss: 1.1958
  step 100/1839, loss: 2.0715
  step 200/1839, loss: 2.7193
  step 300/1839, loss: 0.9174
  step 400/1839, loss: 0.8053
  step 500/1839, loss: 1.6507
  step 600/1839, loss: 0.9236
  step 700/1839, loss: 1.2739
  step 800/1839, loss: 1.3108
  step 900/1839, loss: 1.0938
  step 1000/1839, loss: 0.7243
  step 1100/1839, loss: 1.2123
  step 1200/1839, loss: 0.8148
  step 1300/1839, loss: 1.1835
  step 1400/1839, loss: 0.9822
  step 1500/1839, loss: 1.1311
  step 1600/1839, loss: 1.1896
  step 1700/1839, loss: 0.9339
  step 1800/1839, loss: 0.9600
  train loss: 1.1969 | val loss: 1.3465
  Checkpoint saved: checkpoint_best.pt

Epoch 3/30  (lr=9.00e-08)
  step 0/1839, loss: 0.9892
  step 100/1839, loss: 1.1273
  step 200/1839, loss: 1.0054
  step 300/1839, loss: 1.0797
  step 400/1839, loss: 2.2713
  step 500/1839, loss: 1.0348
  step 600/1839, loss: 0.8828
  s

In [27]:
# ─────────────────────────────────────────
# Fixed loss function with label smoothing
# ─────────────────────────────────────────

def compute_loss(logits, target_ids, loss_mask):
    """Cross-entropy loss with label smoothing on output positions only.
    
    Label smoothing of 0.1 prevents the model from becoming
    overconfident on majority tokens like colour 0.
    """
    logits_flat  = logits.view(-1, VOCAB_SIZE)
    targets_flat = target_ids.view(-1)
    mask_flat    = loss_mask.view(-1)
    
    loss = F.cross_entropy(
        logits_flat[mask_flat],
        targets_flat[mask_flat],
        ignore_index=PAD,
        label_smoothing=0.1  # <-- key fix
    )
    return loss


def train_one_epoch(model, loader, optimizer, scaler, device, grad_accum_steps=4):
    """Train for one epoch with gradient accumulation and BF16 autocast."""
    model.train()
    total_loss    = 0.0
    total_batches = 0
    optimizer.zero_grad()
    
    for step, (inp, tgt, msk) in enumerate(loader):
        inp = inp.to(device)
        tgt = tgt.to(device)
        msk = msk.to(device)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(inp)
            loss   = compute_loss(logits, tgt, msk)
            loss   = loss / grad_accum_steps
        
        scaler.scale(loss).backward()
        
        if (step + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss    += loss.item() * grad_accum_steps
        total_batches += 1
        
        if step % 100 == 0:
            print(f"  step {step}/{len(loader)}, loss: {loss.item()*grad_accum_steps:.4f}")
    
    return total_loss / total_batches


@torch.no_grad()
def evaluate(model, loader, device):
    """Compute average loss on validation set."""
    model.eval()
    total_loss    = 0.0
    total_batches = 0
    
    for inp, tgt, msk in loader:
        inp = inp.to(device)
        tgt = tgt.to(device)
        msk = msk.to(device)
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(inp)
            loss   = compute_loss(logits, tgt, msk)
        
        total_loss    += loss.item()
        total_batches += 1
    
    return total_loss / total_batches


# ─────────────────────────────────────────
# Reinitialize model with higher dropout
# ─────────────────────────────────────────

model = ARCTransformer(
    vocab_size  = VOCAB_SIZE,
    d_model     = 512,
    n_heads     = 8,
    n_layers    = 6,
    d_ff        = 2048,
    max_seq_len = MAX_SEQ_LEN,
    dropout     = 0.3        # increased from 0.1
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# ─────────────────────────────────────────
# Optimizer — higher weight decay
# ─────────────────────────────────────────

NUM_EPOCHS    = 5
WARMUP_EPOCHS = 1
LR            = 3e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.1,    # increased from 0.01
    betas=(0.9, 0.95)
)

def get_lr(epoch):
    """Simple warmup then cosine decay."""
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (NUM_EPOCHS - WARMUP_EPOCHS)
    return max(0.1, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)
scaler    = torch.amp.GradScaler('cuda')

# ─────────────────────────────────────────
# Training loop — only 5 epochs
# ─────────────────────────────────────────

best_val_loss = float('inf')
history       = {'train_loss': [], 'val_loss': []}

print("Retraining with fixes...")
print(f"Epochs: {NUM_EPOCHS}, Dropout: 0.3, Weight decay: 0.1, Label smoothing: 0.1")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    current_lr = scheduler.get_last_lr()[0] * LR
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}  (lr={current_lr:.2e})")
    
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scaler, device,
        grad_accum_steps=4
    )
    val_loss = evaluate(model, val_loader, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    
    print(f"  train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch, val_loss,
                       'checkpoint_best.pt')
        print(f"  New best saved!")

save_checkpoint(model, optimizer, scheduler, NUM_EPOCHS-1, val_loss,
               'checkpoint_final.pt')

print(f"\nDone! Best val loss: {best_val_loss:.4f}")

Model parameters: 26.28M
Retraining with fixes...
Epochs: 5, Dropout: 0.3, Weight decay: 0.1, Label smoothing: 0.1

Epoch 1/5  (lr=9.00e-08)
  step 0/1839, loss: 4.5864
  step 100/1839, loss: 4.5229
  step 200/1839, loss: 2.8713
  step 300/1839, loss: 1.7993
  step 400/1839, loss: 1.8081
  step 500/1839, loss: 1.6638
  step 600/1839, loss: 1.6235
  step 700/1839, loss: 2.1701
  step 800/1839, loss: 1.4680
  step 900/1839, loss: 1.6395
  step 1000/1839, loss: 1.9498
  step 1100/1839, loss: 1.4603
  step 1200/1839, loss: 1.4442
  step 1300/1839, loss: 1.5654
  step 1400/1839, loss: 1.4667
  step 1500/1839, loss: 1.3657
  step 1600/1839, loss: 1.3364
  step 1700/1839, loss: 1.8478
  step 1800/1839, loss: 1.3913
  train loss: 2.1155 | val loss: 1.6893
  Checkpoint saved: checkpoint_best.pt
  New best saved!

Epoch 2/5  (lr=9.00e-08)
  step 0/1839, loss: 1.3712
  step 100/1839, loss: 1.2694
  step 200/1839, loss: 1.4274
  step 300/1839, loss: 2.1345
  step 400/1839, loss: 1.2249
  step 500/

In [19]:
import torch
import os

CKPT_DIR = '/kaggle/working/checkpoints'

# List all saved checkpoints
print("Saved checkpoints:")
for f in sorted(os.listdir(CKPT_DIR)):
    path = os.path.join(CKPT_DIR, f)
    size = os.path.getsize(path) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

# Load the best checkpoint and inspect it
best_ckpt = torch.load(os.path.join(CKPT_DIR, 'checkpoint_best.pt'))
print(f"\nBest checkpoint:")
print(f"  Epoch:     {best_ckpt['epoch'] + 1}")
print(f"  Val loss:  {best_ckpt['val_loss']:.4f}")

Saved checkpoints:
  checkpoint_best.pt  (315.5 MB)
  checkpoint_epoch_10.pt  (315.5 MB)
  checkpoint_epoch_15.pt  (315.5 MB)
  checkpoint_epoch_20.pt  (315.5 MB)
  checkpoint_epoch_25.pt  (315.5 MB)
  checkpoint_epoch_30.pt  (315.5 MB)
  checkpoint_epoch_5.pt  (315.5 MB)
  checkpoint_final.pt  (315.5 MB)

Best checkpoint:
  Epoch:     2
  Val loss:  1.3465


In [28]:
print("Epoch | Train Loss | Val Loss")
print("-" * 35)
for i, (tl, vl) in enumerate(zip(history['train_loss'], history['val_loss'])):
    print(f"  {i+1:2d}  |   {tl:.4f}   |  {vl:.4f}")

Epoch | Train Loss | Val Loss
-----------------------------------
   1  |   2.1155   |  1.6893
   2  |   1.4560   |  1.6401
   3  |   1.3457   |  1.6544
   4  |   1.2612   |  1.6628
   5  |   1.2074   |  1.6601


In [29]:
# Quick check - run inference on 5 training tasks
# and see if predictions are still all zeros

correct = 0
total = 5

for task_id, task in train_tasks[:5]:
    pred_grid, _ = generate_output(model, task, device)
    true_grid = task['test'][0]['output']
    match = pred_grid == true_grid
    
    # Count non-zero predictions
    non_zeros = sum(c for row in pred_grid for c in row if c != 0)
    
    print(f"Task: {task_id}")
    print(f"  Exact match: {match}")
    print(f"  Non-zero cells predicted: {non_zeros}")
    print(f"  First predicted row: {pred_grid[0]}")
    print()
    
    if match:
        correct += 1

print(f"Exact matches: {correct}/{total}")

/tmp/ipykernel_106/1257873810.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


Task: 007bbfb7
  Exact match: False
  Non-zero cells predicted: 0
  First predicted row: [0, 0, 0, 0, 0, 0, 0, 0, 0]

Task: 00d62c1b
  Exact match: False
  Non-zero cells predicted: 0
  First predicted row: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Task: 017c7c7b
  Exact match: False
  Non-zero cells predicted: 18
  First predicted row: [0, 2, 0]

Task: 025d127b
  Exact match: False
  Non-zero cells predicted: 0
  First predicted row: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Task: 045e512c
  Exact match: False
  Non-zero cells predicted: 0
  First predicted row: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Exact matches: 0/5


In [30]:
def evaluate_on_tasks(model, tasks, device, n_attempts=2):
    """Run evaluation on a list of tasks with 2 attempts each.
    
    A task is solved if EITHER attempt exactly matches the target.
    This matches the official ARC evaluation protocol.
    
    Returns:
        results: list of dicts with task_id, solved, attempts
        accuracy: fraction of tasks solved
    """
    model.eval()
    results = []
    solved  = 0
    
    for i, (task_id, task) in enumerate(tasks):
        true_grid = task['test'][0]['output']
        
        # Attempt 1: greedy decoding (temperature=1.0, argmax)
        pred1, _ = generate_output(model, task, device, temperature=1.0)
        
        # Attempt 2: slightly higher temperature for diversity
        pred2, _ = generate_output(model, task, device, temperature=0.8)
        
        attempt1_correct = (pred1 == true_grid)
        attempt2_correct = (pred2 == true_grid)
        task_solved      = attempt1_correct or attempt2_correct
        
        if task_solved:
            solved += 1
        
        results.append({
            'task_id':  task_id,
            'solved':   task_solved,
            'attempt1': attempt1_correct,
            'attempt2': attempt2_correct
        })
        
        # Progress update every 50 tasks
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(tasks)}] solved so far: {solved}")
    
    accuracy = solved / len(tasks)
    return results, accuracy


# ─────────────────────────────────────────
# Run on evaluation tasks
# ─────────────────────────────────────────

print("Running evaluation on 400 eval tasks...")
print("(This will take a few minutes)")
print("=" * 50)

eval_results, eval_accuracy = evaluate_on_tasks(
    model, eval_tasks, device, n_attempts=2
)

print(f"\n{'='*50}")
print(f"FINAL RESULTS")
print(f"{'='*50}")
print(f"Tasks solved:   {sum(r['solved'] for r in eval_results)} / {len(eval_tasks)}")
print(f"Exact match accuracy: {eval_accuracy*100:.2f}%")

# Breakdown
solved_both = sum(1 for r in eval_results if r['attempt1'] and r['attempt2'])
solved_only1 = sum(1 for r in eval_results if r['attempt1'] and not r['attempt2'])
solved_only2 = sum(1 for r in eval_results if not r['attempt1'] and r['attempt2'])

print(f"\nBreakdown:")
print(f"  Solved by attempt 1 only: {solved_only1}")
print(f"  Solved by attempt 2 only: {solved_only2}")
print(f"  Solved by both attempts:  {solved_both}")

# Show which tasks were solved
solved_ids = [r['task_id'] for r in eval_results if r['solved']]
print(f"\nSolved task IDs: {solved_ids}")

Running evaluation on 400 eval tasks...
(This will take a few minutes)


/tmp/ipykernel_106/1257873810.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


  [50/400] solved so far: 0
  [100/400] solved so far: 0
  [150/400] solved so far: 0
  [200/400] solved so far: 1
  [250/400] solved so far: 1
  [300/400] solved so far: 1
  [350/400] solved so far: 1
  [400/400] solved so far: 1

FINAL RESULTS
Tasks solved:   1 / 400
Exact match accuracy: 0.25%

Breakdown:
  Solved by attempt 1 only: 0
  Solved by attempt 2 only: 0
  Solved by both attempts:  1

Solved task IDs: ['7039b2d7']


In [21]:
import os

path = '/kaggle/working/checkpoints'
if os.path.exists(path):
    for f in sorted(os.listdir(path)):
        size = os.path.getsize(f'{path}/{f}') / 1e6
        print(f"{f}  ({size:.1f} MB)")
else:
    print("Checkpoint folder not found — need to retrain")

checkpoint_best.pt  (315.5 MB)
checkpoint_epoch_10.pt  (315.5 MB)
checkpoint_epoch_15.pt  (315.5 MB)
checkpoint_epoch_20.pt  (315.5 MB)
checkpoint_epoch_25.pt  (315.5 MB)
checkpoint_epoch_30.pt  (315.5 MB)
checkpoint_epoch_5.pt  (315.5 MB)
checkpoint_final.pt  (315.5 MB)


In [23]:
import torch
import os

CKPT_DIR = '/kaggle/working/checkpoints'

# Load best checkpoint
ckpt = torch.load(os.path.join(CKPT_DIR, 'checkpoint_best.pt'), 
                  map_location=device)

model.load_state_dict(ckpt['model'])
model.eval()

print(f"Loaded checkpoint from epoch: {ckpt['epoch'] + 1}")
print(f"Val loss: {ckpt['val_loss']:.4f}")
print("Model ready for inference.")

Loaded checkpoint from epoch: 2
Val loss: 1.3465
Model ready for inference.


In [25]:
def tokens_to_grid(tokens, height, width):
    """Convert a flat token list back to a 2D grid.
    
    Ignores ROW_SEP tokens, just reads colour values.
    Returns a 2D list of integers.
    """
    colour_tokens = [t for t in tokens if t not in 
                     (ROW_SEP, GRID_SEP, PAIR_SEP, BOS, EOS, PAD)]
    
    # Truncate or pad to exact grid size
    expected = height * width
    colour_tokens = colour_tokens[:expected]
    while len(colour_tokens) < expected:
        colour_tokens.append(0)  # pad with black if short
    
    grid = []
    for r in range(height):
        grid.append(colour_tokens[r * width : (r + 1) * width])
    return grid


@torch.no_grad()
def generate_output(model, task, device, max_new_tokens=1024, temperature=1.0):
    """Autoregressively generate the output grid for a task.
    
    Feeds the model the full context (demo pairs + test input + GRID_SEP),
    then generates tokens one by one until EOS or max_new_tokens.
    
    Args:
        model:          trained ARCTransformer
        task:           ARC task dict
        device:         torch device
        max_new_tokens: maximum number of tokens to generate
        temperature:    sampling temperature (1.0 = greedy-ish, <1.0 = sharper)
    
    Returns:
        predicted_grid: 2D list of ints
        generated_tokens: raw generated token list
    """
    model.eval()
    
    # Build context: everything up to and including GRID_SEP after test input
    context_tokens = [BOS]
    for pair in task['train']:
        context_tokens.extend(grid_to_tokens(pair['input']))
        context_tokens.append(GRID_SEP)
        context_tokens.extend(grid_to_tokens(pair['output']))
        context_tokens.append(PAIR_SEP)
    context_tokens.extend(grid_to_tokens(task['test'][0]['input']))
    context_tokens.append(GRID_SEP)
    
    # Get expected output shape from ground truth
    # (only used to parse the output, not fed to the model)
    out_h, out_w = get_output_shape(task)
    expected_len = out_h * out_w + out_h  # cells + row separators
    
    input_ids = torch.tensor(
        context_tokens, dtype=torch.long
    ).unsqueeze(0).to(device)  # (1, context_len)
    
    generated = []
    
    for _ in range(min(max_new_tokens, expected_len + 5)):
        # Truncate if exceeding max sequence length
        if input_ids.shape[1] >= MAX_SEQ_LEN:
            break
        
        with autocast(dtype=torch.bfloat16):
            logits = model(input_ids)  # (1, seq_len, vocab_size)
        
        # Take logits at last position
        next_logits = logits[0, -1, :]  # (vocab_size,)
        
        if temperature != 1.0:
            next_logits = next_logits / temperature
        
        # Greedy decoding: pick highest probability token
        next_token = next_logits.argmax(dim=-1).item()
        
        if next_token == EOS:
            break
        
        generated.append(next_token)
        
        # Append to input sequence
        input_ids = torch.cat([
            input_ids,
            torch.tensor([[next_token]], device=device)
        ], dim=1)
    
    predicted_grid = tokens_to_grid(generated, out_h, out_w)
    return predicted_grid, generated


# ─────────────────────────────────────────
# Test on a single training task first
# ─────────────────────────────────────────

task_id, task = train_tasks[0]
pred_grid, gen_tokens = generate_output(model, task, device)

# Ground truth
true_grid = task['test'][0]['output']

print(f"Task: {task_id}")
print(f"Expected output shape: {len(true_grid)}x{len(true_grid[0])}")
print(f"Predicted output shape: {len(pred_grid)}x{len(pred_grid[0])}")
print(f"\nGround truth:")
for row in true_grid:
    print(row)
print(f"\nPredicted:")
for row in pred_grid:
    print(row)
print(f"\nExact match: {pred_grid == true_grid}")

Task: 007bbfb7
Expected output shape: 9x9
Predicted output shape: 9x9

Ground truth:
[7, 0, 7, 0, 0, 0, 7, 0, 7]
[7, 0, 7, 0, 0, 0, 7, 0, 7]
[7, 7, 0, 0, 0, 0, 7, 7, 0]
[7, 0, 7, 0, 0, 0, 7, 0, 7]
[7, 0, 7, 0, 0, 0, 7, 0, 7]
[7, 7, 0, 0, 0, 0, 7, 7, 0]
[7, 0, 7, 7, 0, 7, 0, 0, 0]
[7, 0, 7, 7, 0, 7, 0, 0, 0]
[7, 7, 0, 7, 7, 0, 0, 0, 0]

Predicted:
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]

Exact match: False


/tmp/ipykernel_106/1257873810.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):


In [26]:
# Check what the model's raw probabilities look like
# at the start of generation

task_id, task = train_tasks[0]

# Build context
context_tokens = [BOS]
for pair in task['train']:
    context_tokens.extend(grid_to_tokens(pair['input']))
    context_tokens.append(GRID_SEP)
    context_tokens.extend(grid_to_tokens(pair['output']))
    context_tokens.append(PAIR_SEP)
context_tokens.extend(grid_to_tokens(task['test'][0]['input']))
context_tokens.append(GRID_SEP)

input_ids = torch.tensor(context_tokens, dtype=torch.long).unsqueeze(0).to(device)
print(f"Context length: {input_ids.shape[1]} tokens")
print(f"Max seq len: {MAX_SEQ_LEN}")

model.eval()
with torch.no_grad():
    from torch.cuda.amp import autocast
    with autocast(dtype=torch.bfloat16):
        logits = model(input_ids)

next_logits = logits[0, -1, :]
probs = torch.softmax(next_logits.float(), dim=-1)

print(f"\nProbability distribution over next token:")
for token_id, prob in enumerate(probs):
    token_name = {
        10: 'ROW_SEP', 11: 'GRID_SEP', 12: 'PAIR_SEP',
        13: 'BOS', 14: 'EOS', 15: 'PAD'
    }.get(token_id, str(token_id))
    print(f"  token {token_name:>8}: {prob.item():.4f}")

print(f"\nMost likely next token: {probs.argmax().item()}")
print(f"Context length vs MAX_SEQ_LEN: {input_ids.shape[1]} / {MAX_SEQ_LEN}")

Context length: 534 tokens
Max seq len: 2048

Probability distribution over next token:
  token        0: 0.6372
  token        1: 0.0039
  token        2: 0.0354
  token        3: 0.0061
  token        4: 0.0150
  token        5: 0.0006
  token        6: 0.0977
  token        7: 0.1883
  token        8: 0.0040
  token        9: 0.0101
  token  ROW_SEP: 0.0014
  token GRID_SEP: 0.0001
  token PAIR_SEP: 0.0000
  token      BOS: 0.0000
  token      EOS: 0.0001
  token      PAD: 0.0000

Most likely next token: 0
Context length vs MAX_SEQ_LEN: 534 / 2048


/tmp/ipykernel_106/2834767559.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.bfloat16):
